# Online Retail Customer & Sales Analytics

## Import Libraries

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

## Loading Dataset

In [8]:
df = pd.read_csv(r"D:\Datasets\Online Retail data.csv")
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## Data Understanding

In [9]:
print("Shape:", df.shape)
df.info()
df.describe()

Shape: (541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## Checking Missing Values

In [10]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## Data Cleaning

In [11]:
# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove negative/zero quantity (returns or errors)
df = df[df['Quantity'] > 0]

# Remove negative price
df = df[df['UnitPrice'] > 0]

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

## saving cleaned data
df.to_csv("cleaned_retail_data.csv", index=False)

## Create RFM Features

In [12]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalPrice': 'sum'}).reset_index()                      # Monetary

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

## Feature Engineering

In [13]:
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency'].replace(0, 1)

# Customer Lifetime (FOR ANALYSIS ONLY — NOT MODEL)
first_purchase = df.groupby('CustomerID')['InvoiceDate'].min()
last_purchase = df.groupby('CustomerID')['InvoiceDate'].max()

rfm['Customer_Lifetime'] = (last_purchase - first_purchase).dt.days.values

## Creating Target (Churn)

In [16]:
# Churn: no purchase in last 90 days
rfm['Churn'] = rfm['Recency'].apply(lambda x: 1 if x > 90 else 0)

# Check distribution
print("Churn Distribution:")
print(rfm['Churn'].value_counts())

Churn Distribution:
Churn
0    2889
1    1449
Name: count, dtype: int64


## Prepare Data

In [17]:
X = rfm[['Frequency', 'Monetary', 'AOV']]
y = rfm['Churn']

## TRAIN TEST SPLIT

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Logistic Regression

In [21]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

print("\n=== Logistic Regression ===")
print(classification_report(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.79      0.81      0.80       578
           1       0.61      0.57      0.58       290

    accuracy                           0.73       868
   macro avg       0.70      0.69      0.69       868
weighted avg       0.73      0.73      0.73       868

ROC-AUC: 0.7848824722586802


## Random Forest

In [22]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest ===")
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))


=== Random Forest ===
              precision    recall  f1-score   support

           0       0.73      0.76      0.74       578
           1       0.47      0.43      0.45       290

    accuracy                           0.65       868
   macro avg       0.60      0.60      0.60       868
weighted avg       0.64      0.65      0.64       868

ROC-AUC: 0.662247941773058


## Feature Importance

In [23]:
importance = pd.Series(rf.feature_importances_, index=X.columns)

print("\nFeature Importance:")
print(importance.sort_values(ascending=False))


Feature Importance:
Monetary     0.473088
AOV          0.420509
Frequency    0.106403
dtype: float64


## Churn Probability

In [24]:
test_results = X_test.copy()
test_results['Churn_Probability'] = rf.predict_proba(X_test)[:, 1]
test_results['Actual'] = y_test.values

high_risk = test_results[test_results['Churn_Probability'] > 0.7]

print(high_risk.head())

      Frequency  Monetary         AOV  Churn_Probability  Actual
2004          1    381.32  381.320000               0.74       1
387           1    134.10  134.100000               0.82       1
3667          3    423.89  141.296667               0.86       0
2227          2    596.85  298.425000               0.84       1
1785          3    386.15  128.716667               0.80       1


## Saving output

In [26]:
rfm.to_csv("customer_churn_predictions.csv", index=False)

## Key Insights

- Total revenue ~8.6M from ~19K transactions and ~4.3K customers

- Revenue is concentrated among a small % of customers (Pareto effect)

- Majority of customers are low-frequency or one-time buyers

- High recency (>90 days) + low frequency customers show highest churn risk

- AOV + Frequency provides better behavioral insight than Monetary alone

- Seasonal revenue spikes indicate demand concentration

- Model enables segmentation into Low, Medium, High risk customers

## Conclusion


• RFM features effectively capture customer purchasing behavior

• Model enables prediction of churn probability for each customer

• Risk-based segmentation helps identify high-risk customers for targeted action

• Revenue is highly dependent on a small group of high-value customers

• Improving retention can significantly impact overall revenue

• Model performance improved after removing data leakage, making results more reliable

• Approach supports data-driven decision making for customer retention strategies